#  GPT-5 新参数与新工具

我们在 GPT-5 系列中引入了新的开发者控制项，让你能更好地掌控模型响应——从调整输出长度和风格，到强制执行严格的格式。以下是最新功能的快速概览：


| #  | 功能 | 概述 | 取值 / 用法 |
|----|---------|----------|----------------|
| 1. | **Verbosity 参数** | 让你提示模型在回复中更详尽或更简略。保持提示不变，改用该参数，而不是重写提示。 | • **low** → 简洁的用户体验，文字最少。<br>• **medium** *（默认）* → 细节适中。<br>• **high** → 详尽，非常适合审计、教学或工作交接。 |
| 2. | **自由格式函数调用（Freeform Function Calling）** | 直接向自定义工具生成原始文本负载——从 Python 脚本到 SQL 查询皆可——无需 JSON 封装。为以下外部运行时提供更大的灵活性：<br>• 代码沙箱（Python、C++、Java……）<br>• SQL 数据库<br>• Shell 环境<br>• 配置生成器 | 在不需要结构化 JSON、而原始文本对目标工具更自然时使用。 |
| 3. | **上下文无关文法（CFG）** | 一组定义某种语言中合法字符串的产生式规则。每条规则把一个非终结符改写为终结符和/或其他非终结符，与周围上下文无关。可用于在 OpenAI 工具中约束输出，使其符合编程语言或自定义格式的语法。 | 将其作为契约使用，确保模型只输出文法所接受的合法字符串。 |
| 4. | **最小推理（Minimal Reasoning）** | 以极少或不使用推理 token 的方式运行 GPT-5，以最大限度降低延迟、加快首个 token 的生成。非常适合不需要解释的确定性轻量任务（提取、格式化、简短改写、简单分类）。如果未指定，effort 默认为 medium。 | 设置 reasoning effort："minimal"。避免用于多步骤规划或大量使用工具的工作流。 |


**支持的模型：**  
- gpt-5  
- gpt-5-mini  
- gpt-5-nano  

**支持的 API 端点** 
- Responses API 
- Chat Completions API 

注意：我们建议 GPT-5 系列模型搭配 Responses API 使用，以充分发挥模型性能。 


## 前提条件 

首先，将 OpenAI SDK 更新到支持 GPT-5 新参数和新工具的版本。请确保已将 OPENAI_API_KEY 设置为环境变量。 

In [2]:
!pip install --quiet --upgrade openai pandas && \
echo -n "openai " && pip show openai | grep '^Version:' | cut -d' ' -f2 && \
echo -n "pandas " && pip show pandas | grep '^Version:' | cut -d' ' -f2

openai 1.99.2
pandas 2.3.1


## 1. Verbosity 参数 

### 1.1 概述 
verbosity 参数让你可以提示模型在回复中更详尽或更简略。   

**取值：**"low"、"medium"、"high"

- low → 简洁的用户体验，文字最少。
- medium（默认）→ 细节适中。
- high → 详尽，非常适合审计、教学或工作交接。

保持提示不变，改用该参数，而不是重写提示。


In [3]:
from openai import OpenAI
import pandas as pd
from IPython.display import display

client = OpenAI()

question = "Write a poem about a boy and his first pet dog."

data = []

for verbosity in ["low", "medium", "high"]:
    response = client.responses.create(
        model="gpt-5-mini",
        input=question,
        text={"verbosity": verbosity}
    )

    # Extract text
    output_text = ""
    for item in response.output:
        if hasattr(item, "content"):
            for content in item.content:
                if hasattr(content, "text"):
                    output_text += content.text

    usage = response.usage
    data.append({
        "Verbosity": verbosity,
        "Sample Output": output_text,
        "Output Tokens": usage.output_tokens
    })

# Create DataFrame
df = pd.DataFrame(data)

# Display nicely with centered headers
pd.set_option('display.max_colwidth', None)
styled_df = df.style.set_table_styles(
    [
        {'selector': 'th', 'props': [('text-align', 'center')]},  # Center column headers
        {'selector': 'td', 'props': [('text-align', 'left')]}     # Left-align table cells
    ]
)

display(styled_df)


,Verbosity,Sample Output,Output Tokens
0,low,"He found a scruff of fur behind the shed one spring afternoon, a heartbeat small and fast beneath a coat of dust and light. The world shrank to two—mud on sneakers, a wag, a clumsy tune— names rolled off his tongue like marbles, simple, sure, and bright. They learned the map of each other's hands: the scratch beneath the ear, the way a storm could change the shape of brave into a shake. Mornings were for toast and sunlight, afternoons for running near the riverbank where leaves applauded every leap they'd take. At night they shared a blanket and the secret of the dark, the boy with whispered stories, the dog with steady breath. Years braided into footprints—first skinned knees, then a spark of barnyard gray upon a muzzle, slow and gentle as a wreath. When time unlatched its gates, the boy still carried small things: a collar, a chewed shoe, the echo of a bark that taught him how to hope. He learned that love can look like leaving crumbs of ordinary kings, and that some firsts fit in your pockets long after they have gone.",560
1,medium,"He found him folded in the crook of a cardboard box, a tiny ribcage hitching like a thought. The boy had pockets full of pennies and promises; the dog had eyes like two small questions. They learned names together — the boy said one, the dog tilted his head and accepted it. Mornings were clumsy lessons: leash in hand, the dog discovering sidewalks with a sneeze of wonder, the boy discovering courage at the end of a rope. They chased afternoons into puddles, mud kissing the boy's knees and the dog's whiskers. The dog taught him how to throw sticks that never came back and how to forgive them when they didn't. Evenings were for quiet conspiracies: the dog's breath a warm punctuation against the boy's ankle as the sky grew blue-black. Homework became a small island between their worlds, a pencil, a pat, the faithful presence of paws on carpet. The dog learned how to sit for apples, how to hide a cold nose under a blanket of fingers. The boy learned how to stitch up a torn stuffed bear, how to say sorry and mean it. There were days of thunder when the boy's knees knocked, and the dog, all stern responsibility, pressed his head into the hollow of the boy's fear and held it there as if he could anchor lightning with his chin. They practiced being brave together: doors opened for new schools, new roads, a first bike without training wheels, the dog a steady metronome of tail and warmth, never asking to be anything but present. Seasons unraveled the way they always do. Snow came to lay white questions across the yard; summer stretched its lazy hands and left grass bleaching in August. The boy grew taller and later, the dog moved slower, but in late afternoons they still shared the same light — a private currency of sun and shadow. When the boy learned the language of goodbyes, it was the dog who taught him how to soften them. A last look, a lingering hand across the coat, and a promise that out of all the small ordinary days something invincible had been braided: two hearts, a leash, a map of pawprints on the threshold. Years later, the boy — now grown — tucks a photograph into his coat pocket. He feels the hollow where a warm head used to rest and smiles. Some bonds refuse to be folded away. In the quiet hum of rememberings, he can still hear a collar's jingle and a small, glad bark: first home, first friend, first forever.",849
2,high,"The day the boy met his dog the world grew wider— a small breath of fur and a damp, earnest nose pressed like a secret against his palm. They stood on the porch and the sun tilted curious, as if the sky had come to see how two new things might fit together. He named him after a comic-strip hero, or maybe he didn’t name him at all at first, just laughed and let the sound of it become a name. They learned each other’s weight: the dog’s heavy joy, the boy’s thin, cautious fingers turning into hands that could hold a leaping he

输出 token 数大致随 verbosity 线性增长：low（560）→ medium（849）→ high（1288）。

### 2.3 在编码用例中使用 Verbosity 

verbosity 参数还会影响生成代码的长度和复杂度，以及附带解释的深度。下面是一个示例：我们使用不同的 verbosity 级别，让模型生成一个对 1000000 个随机数组成的数组进行排序的 Python 程序。 

In [4]:
from openai import OpenAI

client = OpenAI()

prompt = "Output a Python program that sorts an array of 1000000 random numbers"

def ask_with_verbosity(verbosity: str, question: str):
    response = client.responses.create(
        model="gpt-5-mini",
        input=question,
        text={
            "verbosity": verbosity
        }
    )

    # Extract assistant's text output
    output_text = ""
    for item in response.output:
        if hasattr(item, "content"):
            for content in item.content:
                if hasattr(content, "text"):
                    output_text += content.text

    # Token usage details
    usage = response.usage

    print("--------------------------------")
    print(f"Verbosity: {verbosity}")
    print("Output:")
    print(output_text)
    print("Tokens => input: {} | output: {}".format(
        usage.input_tokens, usage.output_tokens
    ))


# Example usage:
ask_with_verbosity("low", prompt)

--------------------------------
Verbosity: low
Output:
```python
#!/usr/bin/env python3
import random
import time

def main():
    N = 1_000_000
    arr = [random.random() for _ in range(N)]

    t0 = time.perf_counter()
    arr.sort()
    t1 = time.perf_counter()

    print(f"Sorted {N} numbers in {t1 - t0:.4f} seconds")
    print("First 10:", arr[:10])
    print("Last 10:", arr[-10:])

if __name__ == "__main__":
    main()
```
Tokens => input: 21 | output: 575


注意，代码输出是一个朴素的脚本。现在，我们用 'medium' 来运行 

In [5]:
ask_with_verbosity("medium", prompt)

--------------------------------
Verbosity: medium
Output:
Here's a simple Python script that generates 1,000,000 random numbers, sorts them using the built-in Timsort, and reports timings and a small sample of the sorted output:

```python
#!/usr/bin/env python3
import random
import time

def main():
    N = 1_000_000
    random.seed(42)  # remove or change for different runs

    t0 = time.perf_counter()
    data = [random.random() for _ in range(N)]
    t1 = time.perf_counter()

    data.sort()
    t2 = time.perf_counter()

    # Basic verification and sample output
    is_sorted = all(data[i] <= data[i+1] for i in range(len(data)-1))
    print(f"Generated {N} random numbers in {t1 - t0:.3f} seconds")
    print(f"Sorted in {t2 - t1:.3f} seconds")
    print("Sorted check:", is_sorted)
    print("First 10 values:", data[:10])
    print("Last 10 values:", data[-10:])

if __name__ == "__main__":
    main()
```

Notes:
- This uses Python's built-in list sort (Timsort), which is efficient

medium verbosity 生成了更丰富的代码以及额外的解释。接下来，我们试试 high verbosity。 

In [6]:
ask_with_verbosity("high", prompt)

--------------------------------
Verbosity: high
Output:
Here's a single, self-contained Python program that generates 1,000,000 random numbers and sorts them. It supports two backends: the built-in Python list sort (Timsort) and NumPy (if you have NumPy installed). It measures and prints the time taken for generation, sorting, and verification.

Copy the code into a file (for example sort_random.py) and run it. By default it uses the pure Python backend; pass --backend numpy to use NumPy.

Note: Sorting a million Python floats uses a moderate amount of memory (Python floats and list overhead). NumPy will typically be faster and use less overhead but requires the numpy package.

Program:

import time
import random
import argparse
import sys

def is_sorted_list(a):
    # Linear check for sortedness
    return all(a[i] <= a[i+1] for i in range(len(a)-1))

def main():
    parser = argparse.ArgumentParser(description="Generate and sort random numbers.")
    parser.add_argument("--n", type=

high verbosity 给出了更多细节和解释。 

### 1.3 要点总结 

新的 verbosity 参数能够可靠地调节模型输出的长度和深度，同时保持正确性和推理质量——**无需修改底层提示**。
在本例中：

- **低 verbosity** 生成一个精简、可运行的脚本，没有额外的注释或结构。
- **中 verbosity** 添加了解释性注释、函数结构和可复现性控制。
- **高 verbosity** 生成一个全面的、可用于生产环境的脚本，包含参数解析、多种排序方法、计时/校验、使用说明和最佳实践提示。

## 2. 自由格式函数调用

### 2.1 概述 
现在，GPT‑5 可以使用新的工具类型 `"type": "custom"`，向你的自定义工具发送原始文本负载——从 Python 脚本到 SQL 查询皆可——而无需将数据封装在 JSON 中。这与传统的结构化函数调用不同，让你在与以下外部运行时交互时拥有更大的灵活性：

- 带沙箱的 code_exec（Python、C++、Java……）
- SQL 数据库
- Shell 环境
- 配置生成器

**注意，custom 工具类型不支持并行工具调用。**

### 2.2 快速入门示例——计算圆的面积

下面的代码会生成一段简单的 python 代码来计算圆的面积，并指示模型使用自由格式工具调用来输出结果。 

In [7]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5-mini",
    input="Please use the code_exec tool to calculate the area of a circle with radius equal to the number of 'r's in strawberry",
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "code_exec",
            "description": "Executes arbitrary python code",
        }
    ]
)
print(response.output)

[ResponseReasoningItem(id='rs_6894e31b1f8081999d18325e5aeffcfe0861a2e1728d1664', summary=[], type='reasoning', content=[], encrypted_content=None, status=None), ResponseCustomToolCall(call_id='call_Gnqod2MwPvayp2JdNyA0z0Ah', input='# Counting \'r\'s in the word "strawberry" and computing circle area with that radius\nimport math\nr = "strawberry".count(\'r\')\narea = math.pi * r**2\n{"radius": r, "area": area, "area_exact": f"{r}*pi"}', name='code_exec', type='custom_tool_call', id='ctc_6894e31c66f08199abd622bb5ac3c4260861a2e1728d1664', status='completed')]


模型输出一个包含原始 Python 代码的 `tool call`。你在服务端执行这段代码，捕获打印出的结果，然后在后续的 responses.create 调用中将其发回。

### 2.3 小型基准测试——用三种语言对数组排序
为了演示自由格式工具调用的用法，我们将让 GPT‑5：
- 生成对一个固定数组排序 10 次的 Python、C++ 和 Java 代码。
- 在代码中只打印每次迭代所花费的时间（以毫秒计）。 
- 调用全部三个函数，然后停止 

In [8]:
from openai import OpenAI
from typing import List, Optional

MODEL_NAME = "gpt-5"

# Tools that will be passed to every model invocation. They are defined once so
# that the configuration lives in a single place.
TOOLS = [
    {
        "type": "custom",
        "name": "code_exec_python",
        "description": "Executes python code",
    },
    {
        "type": "custom",
        "name": "code_exec_cpp",
        "description": "Executes c++ code",
    },
    {
        "type": "custom",
        "name": "code_exec_java",
        "description": "Executes java code",
    },
]

client = OpenAI()

def create_response(
    input_messages: List[dict],
    previous_response_id: Optional[str] = None,
):
    """Wrapper around ``client.responses.create``.

    Parameters
    ----------
    input_messages: List[dict]
        The running conversation history to feed to the model.
    previous_response_id: str | None
        Pass the ``response.id`` from the *previous* call so the model can keep
        the thread of the conversation.  Omit on the very first request.
    """
    kwargs = {
        "model": MODEL_NAME,
        "input": input_messages,
        "text": {"format": {"type": "text"}},
        "tools": TOOLS,
    }
    if previous_response_id:
        kwargs["previous_response_id"] = previous_response_id

    return client.responses.create(**kwargs)

# Recursive 
def run_conversation(
    input_messages: List[dict],
    previous_response_id: Optional[str] = None,
):
  
    response = create_response(input_messages, previous_response_id)

    # ``response.output`` is expected to be a list where element 0 is the model
    # message.  Element 1 (if present) denotes a tool call.  When the model is
    # done with tool calls, that element is omitted.
    tool_call = response.output[1] if len(response.output) > 1 else None

    if tool_call and tool_call.type == "custom_tool_call":
        print("--- tool name ---")
        print(tool_call.name)
        print("--- tool call argument (generated code) ---")
        print(tool_call.input)
        
        # Add a synthetic *tool result* so the model can continue the thread.
        
        input_messages.append(
            {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": "done", # <-- replace with the result of the tool call
            }
        )

        # Recurse with updated conversation and track the response id so the
        # model is aware of the prior turn.
        return run_conversation(input_messages, previous_response_id=response.id)
    else:
        # Base-case: no further tool call - return. 
        return 


prompt = """
Write code to sort the array of numbers in three languages: C++, Python and Java (10 times each)using code_exec functions.

ALWAYS CALL THESE THREE FUNCTIONS EXACTLY ONCE: code_exec_python, code_exec_cpp and code_exec_java tools to sort the array in each language. Stop once you've called these three functions in each language once.

Print only the time it takes to sort the array in milliseconds. 

[448, 986, 255, 884, 632, 623, 246, 439, 936, 925, 644, 159, 777, 986, 706, 723, 534, 862, 195, 686, 846, 880, 970, 276, 613, 736, 329, 622, 870, 284, 945, 708, 267, 327, 678, 807, 687, 890, 907, 645, 364, 333, 385, 262, 730, 603, 945, 358, 923, 930, 761, 504, 870, 561, 517, 928, 994, 949, 233, 137, 670, 555, 149, 870, 997, 809, 180, 498, 914, 508, 411, 378, 394, 368, 766, 486, 757, 319, 338, 159, 585, 934, 654, 194, 542, 188, 934, 163, 889, 736, 792, 737, 667, 772, 198, 971, 459, 402, 989, 949]
"""

# Initial developer message.
messages = [
    {
        "role": "developer",
        "content": prompt,
    }
]

run_conversation(messages)


--- tool name ---
code_exec_python
--- tool call argument (generated code) ---
import time

arr = [448, 986, 255, 884, 632, 623, 246, 439, 936, 925, 644, 159, 777, 986, 706, 723, 534, 862, 195, 686, 846, 880, 970, 276, 613, 736, 329, 622, 870, 284, 945, 708, 267, 327, 678, 807, 687, 890, 907, 645, 364, 333, 385, 262, 730, 603, 945, 358, 923, 930, 761, 504, 870, 561, 517, 928, 994, 949, 233, 137, 670, 555, 149, 870, 997, 809, 180, 498, 914, 508, 411, 378, 394, 368, 766, 486, 757, 319, 338, 159, 585, 934, 654, 194, 542, 188, 934, 163, 889, 736, 792, 737, 667, 772, 198, 971, 459, 402, 989, 949]

start = time.perf_counter()
for _ in range(10):
    b = arr[:]  # copy
    b.sort()
elapsed_ms = int((time.perf_counter() - start) * 1000)
print(elapsed_ms, end="")
--- tool name ---
code_exec_cpp
--- tool call argument (generated code) ---
#include <iostream>
#include <vector>
#include <algorithm>
#include <chrono>
using namespace std;

int main() {
    vector<int> a = {448, 986, 255, 884, 632, 6

模型为同一算法输出了 Python、C++ 和 Java 三个代码块。函数调用的输出被串联回模型作为输入，使模型能够持续运行，直到所有函数都恰好被调用一次。 

### 2.4 要点总结 

GPT-5 中的自由格式工具调用让你可以把原始文本负载——例如 Python 脚本、SQL 查询或配置文件——直接发送给自定义工具，而无需 JSON 封装。这为与外部运行时交互提供了更大的灵活性，并让模型能够以工具所期望的确切格式生成代码或文本。在不需要结构化 JSON、而自然文本输出能提升可用性时，这是理想的选择。

## 3. 上下文无关文法（CFG）

### 3.1 概述 
上下文无关文法是一组产生式规则，用于定义哪些字符串属于某种语言。每条规则把一个非终结符改写为由终结符（字面 token）和/或其他非终结符组成的序列，且与周围上下文无关——因此称为“上下文无关”。CFG 可以描述大多数编程语言的语法；在 OpenAI 自定义工具中，它充当契约，强制模型只输出文法所接受的字符串。

### 3.2 文法基础

**支持的文法语法** 
- Lark - https://lark-parser.readthedocs.io/en/stable/
- Regex - https://docs.rs/regex/latest/regex/#syntax

我们在底层使用 LLGuidance 来约束模型采样：https://github.com/guidance-ai/llguidance.

**不支持的 Lark 功能** 
- 正则表达式中的环视（`(?=...)`、`(?!...)` 等）
- 正则表达式中的惰性修饰符（`*?`、`+?`、`??`）。
- 终结符优先级、模板、%declares、%import（%import common 除外）。


**终结符与规则，以及贪婪词法分析** 

| 概念          | 要点                                                                    |
|------------------|------------------------------------------------------------------------------|
| 终结符（大写）| 由词法分析器优先匹配——最长匹配者胜出。                             |
| 规则（小写）    | 组合终结符；无法影响文本如何被切分为 token。                   |
| 贪婪词法分析器     | 切勿试图跨多个终结符“塑造”自由文本——否则你会失去控制。 |

**正确与错误的模式设计** 

✅ **由一个有界终结符处理锚点之间的自由文本**  
```
start: SENTENCE
SENTENCE: /[A-Za-z, ]*(the hero|a dragon)[A-Za-z, ]*(fought|saved)[A-Za-z, ]*(a treasure|the kingdom)[A-Za-z, ]*\./
```
❌ **不要把自由文本拆分到多个终结符/规则中**  
```
start: sentence
sentence: /[A-Za-z, ]+/ subject /[A-Za-z, ]+/ verb /[A-Za-z, ]+/ object /[A-Za-z, ]+/
```

### 3.3 示例——SQL 方言：MS SQL 与 PostgreSQL

以下代码示例现在是使用 CFG 构建多方言 SQL 工具的标准参考。它演示了：

- 两个相互独立的文法定义（`mssql_grammar_definition`、`postgres_grammar_definition`），分别编码 TOP 与 LIMIT 语义。
- 如何在单个脚本中编写提示、调用工具并检查工具调用。
- 并排检查助手的响应。

为不同的 SQL 方言定义 LARK 文法

In [9]:
import textwrap

# ----------------- grammars for MS SQL dialect -----------------
mssql_grammar = textwrap.dedent(r"""
            // ---------- Punctuation & operators ----------
            SP: " "
            COMMA: ","
            GT: ">"
            EQ: "="
            SEMI: ";"

            // ---------- Start ----------
            start: "SELECT" SP "TOP" SP NUMBER SP select_list SP "FROM" SP table SP "WHERE" SP amount_filter SP "AND" SP date_filter SP "ORDER" SP "BY" SP sort_cols SEMI

            // ---------- Projections ----------
            select_list: column (COMMA SP column)*
            column: IDENTIFIER

            // ---------- Tables ----------
            table: IDENTIFIER

            // ---------- Filters ----------
            amount_filter: "total_amount" SP GT SP NUMBER
            date_filter: "order_date" SP GT SP DATE

            // ---------- Sorting ----------
            sort_cols: "order_date" SP "DESC"

            // ---------- Terminals ----------
            IDENTIFIER: /[A-Za-z_][A-Za-z0-9_]*/
            NUMBER: /[0-9]+/
            DATE: /'[0-9]{4}-[0-9]{2}-[0-9]{2}'/
    """)

# ----------------- grammars for PostgreSQL dialect -----------------
postgres_grammar = textwrap.dedent(r"""
            // ---------- Punctuation & operators ----------
            SP: " "
            COMMA: ","
            GT: ">"
            EQ: "="
            SEMI: ";"

            // ---------- Start ----------
            start: "SELECT" SP select_list SP "FROM" SP table SP "WHERE" SP amount_filter SP "AND" SP date_filter SP "ORDER" SP "BY" SP sort_cols SP "LIMIT" SP NUMBER SEMI

            // ---------- Projections ----------
            select_list: column (COMMA SP column)*
            column: IDENTIFIER

            // ---------- Tables ----------
            table: IDENTIFIER

            // ---------- Filters ----------
            amount_filter: "total_amount" SP GT SP NUMBER
            date_filter: "order_date" SP GT SP DATE

            // ---------- Sorting ----------
            sort_cols: "order_date" SP "DESC"

            // ---------- Terminals ----------
            IDENTIFIER: /[A-Za-z_][A-Za-z0-9_]*/
            NUMBER: /[0-9]+/
            DATE: /'[0-9]{4}-[0-9]{2}-[0-9]{2}'/
    """)

### 3.4 生成特定的 SQL 方言 
我们来定义提示，并调用函数生成 MS SQL 方言 

In [10]:
from openai import OpenAI
client = OpenAI()

sql_prompt_mssql = (
    "Call the mssql_grammar to generate a query for Microsoft SQL Server that retrieve the "
    "five most recent orders per customer, showing customer_id, order_id, order_date, and total_amount, "
    "where total_amount > 500 and order_date is after '2025-01-01'. "
)

response_mssql = client.responses.create(
    model="gpt-5",
    input=sql_prompt_mssql,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "mssql_grammar",
            "description": "Executes read-only Microsoft SQL Server queries limited to SELECT statements with TOP and basic WHERE/ORDER BY. YOU MUST REASON HEAVILY ABOUT THE QUERY AND MAKE SURE IT OBEYS THE GRAMMAR.",
            "format": {
                "type": "grammar",
                "syntax": "lark",
                "definition": mssql_grammar
            }
        },
    ],
    parallel_tool_calls=False
)

print("--- MS SQL Query ---")
print(response_mssql.output[1].input)

--- MS SQL Query ---
SELECT TOP 5 customer_id, order_id, order_date, total_amount FROM orders WHERE total_amount > 500 AND order_date > '2025-01-01' ORDER BY order_date DESC;


输出的 SQL 正确使用了 "SELECT TOP" 结构。 

In [11]:
sql_prompt_pg = (
    "Call the postgres_grammar to generate a query for PostgreSQL that retrieve the "
    "five most recent orders per customer, showing customer_id, order_id, order_date, and total_amount, "
    "where total_amount > 500 and order_date is after '2025-01-01'. "
)

response_pg = client.responses.create(
    model="gpt-5",
    input=sql_prompt_pg,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "postgres_grammar",
            "description": "Executes read-only PostgreSQL queries limited to SELECT statements with LIMIT and basic WHERE/ORDER BY. YOU MUST REASON HEAVILY ABOUT THE QUERY AND MAKE SURE IT OBEYS THE GRAMMAR.",
            "format": {
                "type": "grammar",
                "syntax": "lark",
                "definition": postgres_grammar
            }
        },
    ],
    parallel_tool_calls=False,
)

print("--- PG SQL Query ---")
print(response_pg.output[1].input)

--- PG SQL Query ---
SELECT customer_id, order_id, order_date, total_amount FROM orders WHERE total_amount > 500 AND order_date > '2025-01-01' ORDER BY order_date DESC LIMIT 5;


输出凸显了同一逻辑查询的不同物理语法。提供不同的文法，模型就只能为所选方言生成合法的语句。

| 方言       | 生成的查询                                              | 关键区别                          |
|---------------|--------------------------------------------------------------|------------------------------------------|
| MS SQL Server | SELECT TOP 5 customer_id, … ORDER BY order_date DESC;         | 在列列表之前使用 `TOP N` 子句。  |
| PostgreSQL    | SELECT customer_id, … ORDER BY order_date DESC LIMIT 5;       | 在 `ORDER BY` 之后使用 `LIMIT N`。         |



### 3.5 示例——Regex CFG 语法

以下代码示例演示了如何使用 Regex CFG 语法，将自由格式工具调用约束为特定的时间戳模式。

In [12]:
from openai import OpenAI
client = OpenAI()

timestamp_grammar_definition = r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01]) (?:[01]\d|2[0-3]):[0-5]\d$"

timestamp_prompt = (
        "Call the timestamp_grammar to save a timestamp for August 7th 2025 at 10AM."
)

response_mssql = client.responses.create(
    model="gpt-5",
    input=timestamp_prompt,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "timestamp_grammar",
            "description": "Saves a timestamp in date + time in 24-hr format.",
            "format": {
                "type": "grammar",
                "syntax": "regex",
                "definition": timestamp_grammar_definition
            }
        },
    ],
    parallel_tool_calls=False
)

print("--- Timestamp ---")
print(response_mssql.output[1].input)

--- Timestamp ---
2025-08-07 10:00


### 3.5 最佳实践

Lark 文法很难做到完美。简单的文法表现最为可靠，而复杂的文法通常需要对文法定义本身、提示和工具描述进行反复迭代，以确保模型不会偏离分布。

- 保持终结符有界——使用 `/[^.\n]{0,10}*\./` 而不是 `/.*\./`。同时从内容（否定字符类）和长度（`{M,N}` 量词）两方面限制匹配。 
- 优先使用显式字符类，而不是 `.` 通配符。
- 显式处理空白，例如使用 `SP = " "`，而不是全局 `%ignore`。
- 描述你的工具：明确告诉模型 CFG 接受什么，并指示它仔细推理以确保合规。

**故障排除**
- API 因文法过于复杂而拒绝该文法 ➜ 简化规则和终结符，移除 `%ignore.*`。
- 出现意外的 token ➜ 确认终结符之间没有重叠；检查贪婪词法分析器。
- 当模型“偏离分布”时（表现为模型生成过长或重复的输出，语法上合法但语义上错误）：
    - 收紧文法。
    - 迭代提示（添加少样本示例）和工具描述（解释文法并指示模型通过推理来遵循文法）。
    - 尝试更高的 reasoning effort（例如从 medium 提升到 high）。

**资源：**  
- Lark 文档 – https://lark-parser.readthedocs.io/en/stable/
- Lark IDE – https://www.lark-parser.org/ide/
- LLGuidance 语法 – https://github.com/guidance-ai/llguidance/blob/main/docs/syntax.md
- Regex（Rust crate） – https://docs.rs/regex/latest/regex/#syntax

### 3.6 要点总结 

GPT-5 对上下文无关文法（CFG）的支持让你可以严格约束模型输出，使其符合预定义的语法，确保只生成合法的字符串。这对于强制执行编程语言规则或自定义格式尤其有用，可以减少后处理工作和错误。通过提供精确的文法和清晰的工具描述，你可以让模型可靠地保持在目标输出结构之内。

## 4. 最小推理 

### 4.1 概述 

GPT-5 现在支持一种新的最小推理强度（minimal reasoning effort）。使用最小推理强度时，模型只会输出极少的推理 token，或完全不输出。它专为希望极快获得首个用户可见 token 的开发者用例而设计。注意：如果未提供 reasoning effort，默认值为 medium。 

In [13]:
from openai import OpenAI

client = OpenAI()

prompt = "Classify sentiment of the review as positive|neutral|negative. Return one word only." 


response = client.responses.create(
    model="gpt-5",
    input= [{ 'role': 'developer', 'content': prompt }, 
            { 'role': 'user', 'content': 'The food that the restaurant was great! I recommend it to everyone.' }],
    reasoning = {
        "effort": "minimal"
    },
)

# Extract model's text output
output_text = ""
for item in response.output:
    if hasattr(item, "content"):
        for content in item.content:
            if hasattr(content, "text"):
                output_text += content.text

# Token usage details
usage = response.usage

print("--------------------------------")
print("Output:")
print(output_text)


--------------------------------
Output:
positive


### 4.2 要点总结 

最小推理以极少或不使用推理 token 的方式运行 GPT-5，以最大限度降低延迟、加快首个 token 的生成。将其用于不需要解释的确定性轻量任务（提取、格式化、简短改写、简单分类）。如果不指定 effort，默认为 medium——当你更看重速度而非深思熟虑时，请显式设置为 minimal。